In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Silver Orders

In [0]:
log("Reading from Silver orders ...")
df_silver_orders = spark.table(TBL_SILVER_ORDERS)
display(df_silver_orders.limit(10))

## 2. Extract Unique Locations

In [0]:
df_locations = df_silver_orders.select(
    "city", "state", "country", "postal_code", "region"
).distinct()

In [0]:
log(f"Row before cleansing: {df_locations.count():,}")
df_locations = (
    df_locations
    .dropna(subset=["city", "state", "postal_code"])
    .dropDuplicates(["city", "state", "postal_code"])
    )
log(f"Row after cleansing: {df_locations.count():,}")

display(df_locations.orderBy("city"))


## 3. Generate Surrogate Key

In [0]:
from pyspark.sql.window import Window

df_dim_location = df_locations.withColumn(
    "location_id",
    F.dense_rank().over(
        Window.partitionBy(F.col("country"))
               .orderBy(F.col("city"), F.col("state"), F.col("postal_code"))
    ),  
).select(
    "location_id",
    "city",
    "state",
    "country",
    "postal_code",
    "region"
)

log(f"Total location records: {df_dim_location.count():,}")

display(df_dim_location.orderBy("location_id"))

## 4. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_DIM_LOCATION):
    log(f"Upserting into {TBL_GOLD_DIM_LOCATION} ...")

    delta_table = DeltaTable.forName(spark, TBL_GOLD_DIM_LOCATION)
    (
        delta_table.alias("target")
        .merge(
            df_dim_location.alias("source"),
            """
            target.city = source.city AND
            target.state = source.state AND
            target.postal_code = source.postal_code
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_LOCATION} upserted")

else:
    log(f"Creating {TBL_GOLD_DIM_LOCATION} ...")
    (
        df_dim_location.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(TBL_GOLD_DIM_LOCATION)
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_LOCATION} created")
    

In [0]:
display(spark.table(TBL_GOLD_DIM_LOCATION))